In [ ]:
# ========================================
# CELL 1: INSTALL REQUIRED LIBRARIES
# ========================================
!pip install --upgrade azure-storage-blob pandas scikit-learn imbalanced-learn seaborn matplotlib python-dotenv joblib

# ========================================
# CELL 2: IMPORT LIBRARIES & CONFIGURE CREDENTIALS (DEMO ONLY!)
# ========================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from azure.storage.blob import BlobServiceClient
from io import BytesIO

# ⚠️ SECURITY RISK: Hardcoded credentials for DEMO ONLY
# In production: Use environment variables + Azure Key Vault
CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=fraudstorage1;AccountKey=EO13DTUhbp9OxLYJSCuVt3YG2ZKlEf+jjVwxirVVI0JPpgRhS0WG+OuYnncLcOVrzAatA6ueSB9k+AStHP6vGg==;EndpointSuffix=core.windows.net"
CONTAINER_NAME = "creditcardblob"
BLOB_NAME = "creditcard_dataset.csv"  # As provided (.csv extension)

print("✅ Configuration loaded")
print(f"Storage Account: fraudstroage1")
print(f"Container: {CONTAINER_NAME}")
print(f"Blob: {BLOB_NAME}")

# ========================================
# CELL 3: DOWNLOAD DATASET FROM AZURE BLOB STORAGE
# ========================================
print("\n📥 Downloading dataset from Azure Blob Storage...")
try:
    # Connect to storage account
    blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)
    blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob=BLOB_NAME)
    
    # Download blob content
    blob_data = blob_client.download_blob().readall()
    print(f"✅ Downloaded {len(blob_data):,} bytes")
    
    # Load into pandas DataFrame
    df = pd.read_csv(BytesIO(blob_data))
    print(f"\n✅ Dataset loaded successfully!")
    print(f"📊 Shape: {df.shape}")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nFirst 5 rows:\n{df.head(5)}")
    
except Exception as e:
    print(f"❌ Error downloading blob: {str(e)}")
    print("\n⚠️ Trying with .csv extension (common issue)...")
    try:
        # Fallback: Try with .csv extension
        BLOB_NAME_CSV = "creditcard_dataset_csv.csv"
        blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob=BLOB_NAME_CSV)
        blob_data = blob_client.download_blob().readall()
        df = pd.read_csv(BytesIO(blob_data))
        print(f"✅ Successfully loaded with .csv extension!")
        print(f"Shape: {df.shape}")
    except Exception as e2:
        print(f"❌ Both attempts failed. Verify blob name in Azure Portal:")
        print(f"   https://portal.azure.com → Storage Accounts → fraudstorage1 → Containers → {CONTAINER_NAME}")
        raise

# ========================================
# CELL 4: EXPLORATORY DATA ANALYSIS (EDA)
# ========================================
print("\n" + "="*60)
print("FRAUD CLASS DISTRIBUTION ANALYSIS")
print("="*60)

# Check if 'Class' column exists (standard in credit card fraud datasets)
if 'Class' not in df.columns:
    print("⚠️ Warning: 'Class' column not found!")
    print("Available columns:", list(df.columns))
    print("\n💡 TIP: If your target column has a different name (e.g., 'is_fraud'), update code in Cell 5 & 7")
    # For demo, assume last column is target (common in some datasets)
    target_col = df.columns[-1]
    print(f"Using '{target_col}' as target column for demo purposes")
    df = df.rename(columns={target_col: 'Class'})

# Show class distribution
fraud_count = df['Class'].sum()
total_count = len(df)
fraud_pct = (fraud_count / total_count) * 100

print(f"Total Transactions: {total_count:,}")
print(f"Fraud Cases:        {fraud_count:,} ({fraud_pct:.4f}%)")
print(f"Legitimate Cases:   {total_count - fraud_count:,} ({100 - fraud_pct:.4f}%)")

# Visualize imbalance
plt.figure(figsize=(8, 5))
sns.countplot(x='Class', data=df, palette=['#28a745', '#dc3545'])
plt.title('Transaction Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Class (0=Legitimate, 1=Fraud)')
plt.ylabel('Count')
plt.xticks([0, 1], ['Legitimate', 'Fraud'])
for i, v in enumerate(df['Class'].value_counts().sort_index()):
    plt.text(i, v + (total_count * 0.01), f'{v:,}', ha='center', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.show()

# Check data quality
print("\n" + "="*60)
print("DATA QUALITY CHECK")
print("="*60)
missing = df.isnull().sum()
if missing.sum() == 0:
    print("✅ No missing values found!")
else:
    print("⚠️ Missing values detected:")
    print(missing[missing > 0])

# ========================================
# CELL 5: PREPROCESSING & HANDLE IMBALANCE WITH SMOTE
# ========================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

print("\n⚙️ Preprocessing data...")
# Separate features and target
X = df.drop('Class', axis=1)
y = df['Class']

# Train-test split (stratified to preserve fraud ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Training set:   {X_train.shape[0]:,} samples ({y_train.sum()} fraud cases)")
print(f"Test set:       {X_test.shape[0]:,} samples ({y_test.sum()} fraud cases)")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE to training data ONLY
print("\n🔄 Applying SMOTE to balance training data...")
smote = SMOTE(random_state=42, sampling_strategy='minority')
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f"Before SMOTE:   {np.bincount(y_train)}")
print(f"After SMOTE:    {np.bincount(y_train_resampled)}")
print(f"Resampled training set shape: {X_train_resampled.shape}")

# ========================================
# CELL 6: TRAIN RANDOM FOREST CLASSIFIER
# ========================================
from sklearn.ensemble import RandomForestClassifier

print("\n🌲 Training Random Forest model...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

rf_model.fit(X_train_resampled, y_train_resampled)
print("✅ Training completed!")

# ========================================
# CELL 7: EVALUATE MODEL WITH FRAUD-FOCUSED METRICS
# ========================================
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    roc_auc_score, precision_recall_curve, auc, ConfusionMatrixDisplay
)

# Generate predictions
y_pred = rf_model.predict(X_test_scaled)
y_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

# Classification report
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))

# Confusion matrix
plt.figure(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Fraud'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

# Critical fraud metrics
tn, fp, fn, tp = cm.ravel()
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("="*60)
print("CRITICAL FRAUD DETECTION METRICS")
print("="*60)
print(f"Recall (Fraud Detection Rate):    {recall:.4f} → {recall*100:.2f}% of actual frauds caught")
print(f"Precision (Alert Accuracy):       {precision:.4f} → {precision*100:.2f}% of fraud alerts are real")
print(f"F1-Score:                         {f1:.4f}")
print(f"False Negatives (Missed Fraud):   {fn:,} transactions ⚠️")
print(f"ROC-AUC:                          {roc_auc_score(y_test, y_proba):.4f}")

# Precision-Recall curve
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_proba)
pr_auc = auc(recall_vals, precision_vals)

plt.figure(figsize=(8, 5))
plt.plot(recall_vals, precision_vals, color='#007bff', linewidth=2, label=f'PR-AUC = {pr_auc:.4f}')
plt.axhline(y=y_test.mean(), color='r', linestyle='--', label=f'Baseline = {y_test.mean():.4f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Critical for Imbalanced Fraud Data)', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ========================================
# CELL 8: FEATURE IMPORTANCE ANALYSIS
# ========================================
print("\n" + "="*60)
print("TOP 10 FEATURES INFLUENCING FRAUD PREDICTION")
print("="*60)

importances = rf_model.feature_importances_
feature_names = X.columns
indices = np.argsort(importances)[::-1][:10]

for rank, idx in enumerate(indices, 1):
    print(f"{rank:2d}. {feature_names[idx]:25s} → {importances[idx]:.4f}")

plt.figure(figsize=(10, 6))
plt.barh(range(10), importances[indices][::-1], align="center", color='#17a2b8')
plt.yticks(range(10), [feature_names[i] for i in indices][::-1])
plt.xlabel("Relative Importance")
plt.title("Top 10 Features Influencing Fraud Prediction", fontweight='bold')
plt.tight_layout()
plt.show()
